# Cell 1: download your repo

In [ ]:
!rm -rf /kaggle/working/Gemma4_FineTune_Creativity
!rm -rf /kaggle/working/Gemma4_FineTune_Creativity-main
!rm -f /kaggle/working/repo.zip

!wget -qO /kaggle/working/repo.zip https://github.com/AlinBolcas/Gemma4_FineTune_Creativity/archive/refs/heads/main.zip
!unzip -q /kaggle/working/repo.zip -d /kaggle/working
!rm -f /kaggle/working/repo.zip
!mv /kaggle/working/Gemma4_FineTune_Creativity-main /kaggle/working/Gemma4_FineTune_Creativity

# Cell 2: move into repo

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path("/kaggle/working/Gemma4_FineTune_Creativity")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print(ROOT)

# Cell 3: install training deps

In [ ]:
!pip install -q -U unsloth datasets trl peft accelerate bitsandbytes huggingface_hub sentencepiece
!pip install -q "protobuf>=5.26,<6"

# Cell 4: load Hugging Face token

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
os.environ["HUGGINGFACE_ACCESS_TOKEN"] = hf_token

print("HF token loaded")

# Cell 5: verify GPU

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Cell 6: build a fresh Kaggle config
This makes a new config using Kaggle paths, not your Mac paths.

In [ ]:
from src.III_fineTune.sft_train import discover_dataset_bundles

bundles = discover_dataset_bundles()
for b in bundles:
    print(
        b["base"],
        "train=", b["train_count"],
        "eval=", b["eval_count"],
        "test=", b["test_count"],
    )

bundles

In [ ]:
from src.III_fineTune.sft_train import (
    discover_dataset_bundles,
    build_run_config,
    save_run_config,
    run_preflight,
)

bundles = discover_dataset_bundles()
if not bundles:
    raise RuntimeError("No dataset bundles found. Run format_sft first.")

# Pick the largest formatted bundle, expected to be your final all-seeds dataset
bundle = sorted(bundles, key=lambda b: b["train_count"], reverse=True)[0]
train_count = int(bundle["train_count"])
eval_count = int(bundle["eval_count"])
test_count = int(bundle["test_count"])

print("Using bundle:", bundle["base"])
print("train / eval / test:", train_count, eval_count, test_count)

config = build_run_config(
    bundle=bundle,
    model_alias="e4b",
    backend="unsloth",
    run_name=f"{bundle['base']}_e4b_final",
)

# Runtime
config["runtime"]["target"] = "kaggle"
config["runtime"]["post_train_eval"] = False
config["runtime"]["preflight_sample_count"] = 5

# Output
config["training"]["output_dir"] = f"data/output/models/{bundle['base']}_e4b_final"

# Good final defaults for E4B LoRA on Kaggle
config["training"]["max_steps"] = 0
config["training"]["per_device_train_batch_size"] = 1
config["training"]["gradient_accumulation_steps"] = 4
config["training"]["max_seq_length"] = 4096

config["training"]["lora_r"] = 16
config["training"]["lora_alpha"] = 32
config["training"]["lora_dropout"] = 0.05

config["training"]["weight_decay"] = 0.01
config["training"]["warmup_steps"] = 10

# Slightly adapt epochs/LR to dataset size
if train_count < 100:
    config["training"]["num_train_epochs"] = 8
    config["training"]["learning_rate"] = 1.5e-4
elif train_count < 300:
    config["training"]["num_train_epochs"] = 5
    config["training"]["learning_rate"] = 1e-4
else:
    config["training"]["num_train_epochs"] = 4
    config["training"]["learning_rate"] = 1.5e-4

config_path = save_run_config(config)
print("Saved config:", config_path)

summary = run_preflight(config)
summary

# Cell 7: train

In [ ]:
from src.III_fineTune.sft_train import train_from_config
train_from_config(config)

# ZIP Fine Tuned Adaptor

In [ ]:
run_name = config['run_name']
!cd /kaggle/working/Gemma4_FineTune_Creativity && zip -r /kaggle/working/{run_name}_adapter.zip data/output/models/{run_name}
!cd /kaggle/working/Gemma4_FineTune_Creativity && zip -r /kaggle/working/{run_name}_stats.zip data/output/training_runs/{run_name}